# CoT Injection Experiment

Tests whether a model's Chain-of-Thought trace is a genuine computational scaffold or post-hoc rationalization
by injecting wrong intermediate answers mid-generation and measuring whether the model recovers, propagates the
error, or collapses. Combined with linear probing of hidden states, the central finding is a **dissociation**:
cases where the model's internal representations encode the correct answer, but its output still follows the
injected wrong token.

**Supported models:** `Qwen/Qwen2.5-3B-Instruct`, `Qwen/Qwen2.5-7B-Instruct`

In [ ]:
# Install dependencies
!pip install -q 'transformers>=4.40.0' accelerate datasets torch scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import os
import re
import random
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
)

warnings.filterwarnings('ignore')
print('Imports OK. PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
@dataclass
class ExperimentConfig:
    # Supported: Qwen/Qwen2.5-3B-Instruct, Qwen/Qwen2.5-7B-Instruct
    model_name: str = 'Qwen/Qwen2.5-7B-Instruct'
    load_in_4bit: bool = False  # Not needed for <=7B
    load_in_8bit: bool = False  # Not needed for <=7B
    dataset: str = 'gsm8k'     # 'gsm8k' or 'prontoqa'
    n_problems: int = 200
    difficulty_bins: int = 3
    max_cot_tokens: int = 512
    max_answer_tokens: int = 200
    temperature: float = 0.0
    injection_points: List[str] = field(default_factory=lambda: ['early', 'middle', 'late'])
    error_types: List[str] = field(default_factory=lambda: [
        'near', 'far', 'self_consistent', 'constraint_violating', 'correct_rephrased'
    ])
    run_probing: bool = True
    output_dir: str = './results'

cfg = ExperimentConfig()
print(cfg)

## Section 1: Model Loading

In [ ]:
def load_model(cfg: ExperimentConfig):
    """Load model and tokenizer. Supports 3B and 7B Qwen2.5 Instruct models."""
    supported = ['Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']
    if cfg.model_name not in supported:
        raise ValueError(f'Only {supported} are supported. Got: {cfg.model_name}')

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        output_hidden_states=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f'Loaded {cfg.model_name}')
    print(f'Parameters: {n_params:.2f}B')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f'GPU {i}: {alloc:.1f}GB allocated / {total:.1f}GB total')

    return model, tokenizer

## Section 2: Dataset Loading and Difficulty Stratification

In [ ]:
def _stratify_and_sample(examples: List[dict], cfg: ExperimentConfig) -> List[dict]:
    difficulties = [e['difficulty'] for e in examples]
    bin_edges = np.percentile(difficulties, np.linspace(0, 100, cfg.difficulty_bins + 1))
    bin_edges[-1] += 1  # ensure max value falls in last bin
    for e in examples:
        idx = int(np.searchsorted(bin_edges[1:], e['difficulty']))
        e['difficulty_bin'] = min(idx, cfg.difficulty_bins - 1)

    per_bin = cfg.n_problems // cfg.difficulty_bins
    rng = np.random.default_rng(42)
    sampled = []
    for b in range(cfg.difficulty_bins):
        bucket = [e for e in examples if e['difficulty_bin'] == b]
        n = min(per_bin, len(bucket))
        chosen = rng.choice(len(bucket), n, replace=False)
        sampled.extend([bucket[i] for i in chosen])

    rng.shuffle(sampled)
    for i, e in enumerate(sampled):
        e['id'] = i
    return sampled


def _load_gsm8k(cfg: ExperimentConfig) -> List[dict]:
    ds = load_dataset('gsm8k', 'main', split='test')
    examples = []
    for item in ds:
        solution = item['answer']
        steps = [l for l in solution.split('\n')
                 if l.strip() and not l.strip().startswith('####')]
        difficulty = len(steps)
        m = re.search(r'####\s*([\d,]+)', solution)
        gold_str = m.group(1).replace(',', '') if m else None
        gold_float = float(gold_str) if gold_str else None
        examples.append({
            'question': item['question'],
            'solution': solution,
            'difficulty': difficulty,
            'gold_answer_str': gold_str,
            'gold_answer': gold_float,
        })
    return examples


def _load_prontoqa(cfg: ExperimentConfig) -> List[dict]:
    ds = load_dataset('EleutherAI/prontoqa', split='test')
    examples = []
    for item in ds:
        proof = item.get('proof', item.get('chain_of_thought', ''))
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', proof.strip()) if s.strip()]
        difficulty = len(sentences)
        gold = str(item.get('answer', item.get('label', ''))).strip().lower()
        examples.append({
            'question': item['question'],
            'proof': proof,
            'difficulty': difficulty,
            'gold_answer_str': gold,
            'gold_answer': gold,
        })
    return examples


def load_examples(cfg: ExperimentConfig) -> List[dict]:
    if cfg.dataset == 'gsm8k':
        raw = _load_gsm8k(cfg)
    elif cfg.dataset == 'prontoqa':
        raw = _load_prontoqa(cfg)
    else:
        raise ValueError(f'Unknown dataset: {cfg.dataset}')
    examples = _stratify_and_sample(raw, cfg)
    print(f'Loaded {len(examples)} examples from {cfg.dataset}')
    bins = {b: sum(1 for e in examples if e['difficulty_bin'] == b) for b in range(cfg.difficulty_bins)}
    print('Difficulty bin counts:', bins)
    return examples

## Section 3: Prompting Infrastructure

In [ ]:
SYSTEM_PROMPTS = {
    'gsm8k': (
        'Solve the following math problem step by step. '
        'End with: The answer is [NUMBER].'
    ),
    'prontoqa': (
        'Reason through the following problem step by step. '
        'End with: The answer is [True/False].'
    ),
}


def format_messages(question: str, cfg: ExperimentConfig) -> List[dict]:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPTS[cfg.dataset]},
        {'role': 'user', 'content': question},
    ]


class StopOnSubstring(StoppingCriteria):
    def __init__(self, stop_strings: List[str], tokenizer):
        self.stop_strings = [s.lower() for s in stop_strings]
        self.tokenizer = tokenizer

    def __call__(self, input_ids, scores, **kwargs):
        # Only decode the last 100 tokens to avoid false positives from the prompt
        last = input_ids[0, -100:]
        decoded = self.tokenizer.decode(last, skip_special_tokens=True).lower()
        return any(s in decoded for s in self.stop_strings)


def generate_with_hidden_states(
    model,
    tokenizer,
    prompt,
    max_new_tokens: int,
    temperature: float = 0.0,
    stop_strings: Optional[List[str]] = None,
    return_hidden_states: bool = False,
    prompt_is_string: bool = False,
) -> Tuple[str, Optional[List[np.ndarray]]]:
    """Generate text, optionally returning per-layer hidden states at the first generated token."""
    if prompt_is_string:
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    else:
        text = tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

    stopping_criteria = None
    if stop_strings:
        stopping_criteria = StoppingCriteriaList(
            [StopOnSubstring(stop_strings, tokenizer)]
        )

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        stopping_criteria=stopping_criteria,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=(temperature > 0),
    )
    if temperature > 0:
        gen_kwargs['temperature'] = temperature
    if return_hidden_states:
        gen_kwargs['return_dict_in_generate'] = True
        gen_kwargs['output_hidden_states'] = True

    with torch.inference_mode():
        output = model.generate(**inputs, **gen_kwargs)

    n_input = inputs['input_ids'].shape[1]

    if return_hidden_states:
        generated_ids = output.sequences[0][n_input:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        # output.hidden_states[token_idx][layer_idx] has shape (batch, seq_len, hidden_dim)
        hidden_states = None
        if output.hidden_states and len(output.hidden_states) > 0:
            first_tok = output.hidden_states[0]  # per-layer tensors for first generated token
            hidden_states = [layer[0, -1, :].float().cpu().numpy() for layer in first_tok]
        return generated_text, hidden_states
    else:
        generated_ids = output[0][n_input:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return generated_text, None

## Section 4: CoT Segmentation

In [ ]:
def segment_cot_gsm8k(text: str) -> List[str]:
    steps = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        if re.search(r'the answer is', line, re.IGNORECASE):
            continue
        steps.append(line)
    return steps


def segment_cot_prontoqa(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    steps = []
    for s in parts:
        s = s.strip()
        if not s:
            continue
        if re.search(r'the answer is', s, re.IGNORECASE):
            continue
        steps.append(s)
    return steps


def get_injection_prefix(
    steps: List[str], injection_point: str
) -> Tuple[Optional[str], Optional[int]]:
    n = len(steps)
    if n < 2:
        return None, None

    if injection_point == 'early':
        idx = max(1, n // 4)
    elif injection_point == 'middle':
        idx = n // 2
    elif injection_point == 'late':
        idx = max(n - 2, n // 2 + 1)
    else:
        raise ValueError(f'Unknown injection_point: {injection_point}')

    idx = min(idx, n - 1)  # clamp so steps[idx] is valid
    prefix = '\n'.join(steps[:idx])
    return prefix, idx


def extract_numeric_from_step(step: str) -> Optional[float]:
    matches = re.findall(r'-?[\d,]+\.?\d*', step)
    if not matches:
        return None
    try:
        return float(matches[-1].replace(',', ''))
    except ValueError:
        return None

## Section 5: Wrong Answer Generation

In [ ]:
_rng = random.Random(42)


def _replace_last_number(text: str, replacement: str) -> Optional[str]:
    matches = list(re.finditer(r'-?[\d,]+\.?\d*', text))
    if not matches:
        return None
    m = matches[-1]
    return text[:m.start()] + replacement + text[m.end():]


def generate_wrong_answer_gsm8k(
    correct_value: float,
    gold_answer: float,
    error_type: str,
    step_text: str,
) -> Tuple[Optional[float], Optional[str]]:
    if extract_numeric_from_step(step_text) is None:
        return None, None

    if error_type == 'near':
        factor = _rng.choice([0.80, 0.85, 1.15, 1.20])
        wrong = correct_value * factor
    elif error_type == 'far':
        factor = _rng.choice([10, 0.1, 100])
        wrong = correct_value * factor
    elif error_type == 'self_consistent':
        wrong = gold_answer * _rng.uniform(0.4, 0.6)
    elif error_type == 'constraint_violating':
        wrong = -abs(correct_value * 0.5)
    elif error_type == 'correct_rephrased':
        wrong = correct_value
    else:
        raise ValueError(f'Unknown error_type: {error_type}')

    wrong_str = str(int(wrong)) if wrong == int(wrong) else f'{wrong:.2f}'
    injection_text = _replace_last_number(step_text, wrong_str)
    if injection_text is None:
        return None, None
    return wrong, injection_text


def generate_wrong_answer_prontoqa(
    step_text: str,
    error_type: str,
    gold_answer: str,
) -> Tuple[None, Optional[str]]:
    if error_type == 'correct_rephrased':
        return None, step_text
    elif error_type in ('near', 'self_consistent'):
        return None, re.sub(r'\bis\b', 'is not', step_text, count=1)
    elif error_type in ('far', 'constraint_violating'):
        return None, 'Therefore, the opposite conclusion must hold here.'
    else:
        raise ValueError(f'Unknown error_type: {error_type}')

## Section 6: Answer Extraction and Outcome Classification

In [ ]:
def extract_final_answer_gsm8k(text: str) -> Optional[float]:
    for pattern in [
        r'[Tt]he answer is\s*\$?\s*(-?[\d,]+\.?\d*)',
        r'####\s*(-?[\d,]+\.?\d*)',
    ]:
        m = re.search(pattern, text)
        if m:
            try:
                return float(m.group(1).replace(',', ''))
            except ValueError:
                pass
    # Fallback: last number
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(nums):
        try:
            return float(n.replace(',', ''))
        except ValueError:
            continue
    return None


def extract_final_answer_prontoqa(text: str) -> Optional[str]:
    m = re.search(r'[Tt]he answer is\s*(True|False)', text, re.IGNORECASE)
    if m:
        return m.group(1).lower()
    matches = list(re.finditer(r'\b(True|False)\b', text, re.IGNORECASE))
    if matches:
        return matches[-1].group(1).lower()
    return None


def classify_recovery(
    model_answer,
    gold_answer,
    injected_wrong,
    error_type: str,
    dataset: str,
) -> str:
    if error_type == 'correct_rephrased':
        if dataset == 'gsm8k':
            if model_answer is not None and gold_answer is not None:
                if abs(model_answer - gold_answer) / (abs(gold_answer) + 1e-9) < 0.01:
                    return 'control_correct'
            return 'control_incorrect'
        else:
            if model_answer is not None and str(model_answer).lower() == str(gold_answer).lower():
                return 'control_correct'
            return 'control_incorrect'

    if dataset == 'gsm8k':
        if model_answer is None:
            return 'collapse'
        if gold_answer is not None:
            if abs(model_answer - gold_answer) / (abs(gold_answer) + 1e-9) < 0.01:
                return 'full_recovery'
        if injected_wrong is not None and abs(injected_wrong) > 1e-9:
            if abs(model_answer - injected_wrong) / abs(injected_wrong) < 0.05:
                return 'propagation'
        return 'collapse'
    else:  # prontoqa
        if model_answer is None:
            return 'collapse'
        if str(model_answer).lower() == str(gold_answer).lower():
            return 'full_recovery'
        return 'propagation'

## Section 7: Core Experiment Runner

In [ ]:
@dataclass
class ExperimentResult:
    problem_id: int
    difficulty: int
    difficulty_bin: int
    injection_point: str
    error_type: str
    n_cot_steps: int
    injection_step_idx: Optional[int]
    correct_step_value: Optional[float]
    injected_wrong_value: Optional[float]
    injected_text: Optional[str]
    model_output: str
    model_answer: Optional[Any]
    gold_answer: Optional[Any]
    outcome: str
    hidden_states: Optional[List[np.ndarray]] = None


def run_single_injection(
    model,
    tokenizer,
    example: dict,
    cfg: ExperimentConfig,
    injection_point: str,
    error_type: str,
) -> Optional[ExperimentResult]:
    try:
        messages = format_messages(example['question'], cfg)

        # Generate baseline CoT to obtain step structure
        baseline_text, _ = generate_with_hidden_states(
            model, tokenizer, messages,
            max_new_tokens=cfg.max_cot_tokens,
            temperature=cfg.temperature,
            return_hidden_states=False,
        )

        # Segment into steps
        if cfg.dataset == 'gsm8k':
            steps = segment_cot_gsm8k(baseline_text)
        else:
            steps = segment_cot_prontoqa(baseline_text)

        if len(steps) < 2:
            return None

        # Get injection position
        prefix, step_idx = get_injection_prefix(steps, injection_point)
        if prefix is None:
            return None

        target_step = steps[step_idx]
        correct_value = extract_numeric_from_step(target_step)
        gold_answer = example['gold_answer']

        # Generate wrong injection
        if cfg.dataset == 'gsm8k':
            wrong_value, injection_text = generate_wrong_answer_gsm8k(
                correct_value or 0.0, gold_answer or 0.0, error_type, target_step
            )
        else:
            wrong_value, injection_text = generate_wrong_answer_prontoqa(
                target_step, error_type, example['gold_answer_str']
            )

        if injection_text is None:
            return None

        # Build injected prompt as a string
        original_prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prefix_part = (prefix + '\n') if prefix else ''
        injected_prompt = original_prompt_text + prefix_part + injection_text + '\n'

        # Generate continuation from injected prompt
        model_output, hidden_states = generate_with_hidden_states(
            model, tokenizer, injected_prompt,
            max_new_tokens=cfg.max_answer_tokens,
            temperature=cfg.temperature,
            return_hidden_states=cfg.run_probing,
            prompt_is_string=True,
        )

        # Extract and classify
        if cfg.dataset == 'gsm8k':
            model_answer = extract_final_answer_gsm8k(model_output)
        else:
            model_answer = extract_final_answer_prontoqa(model_output)

        outcome = classify_recovery(
            model_answer, gold_answer, wrong_value, error_type, cfg.dataset
        )

        return ExperimentResult(
            problem_id=example['id'],
            difficulty=example['difficulty'],
            difficulty_bin=example['difficulty_bin'],
            injection_point=injection_point,
            error_type=error_type,
            n_cot_steps=len(steps),
            injection_step_idx=step_idx,
            correct_step_value=correct_value,
            injected_wrong_value=wrong_value,
            injected_text=injection_text,
            model_output=model_output,
            model_answer=model_answer,
            gold_answer=gold_answer,
            outcome=outcome,
            hidden_states=hidden_states,
        )

    except Exception as exc:
        print(f'  Error (problem={example.get("id","?")}, {injection_point}, {error_type}): {exc}')
        return None


def save_results(results: List[ExperimentResult], cfg: ExperimentConfig, checkpoint: bool = False):
    os.makedirs(cfg.output_dir, exist_ok=True)
    suffix = '_checkpoint' if checkpoint else ''

    rows = []
    hs_dict: Dict[str, np.ndarray] = {}
    for r in results:
        rows.append({
            'problem_id': r.problem_id,
            'difficulty': r.difficulty,
            'difficulty_bin': r.difficulty_bin,
            'injection_point': r.injection_point,
            'error_type': r.error_type,
            'n_cot_steps': r.n_cot_steps,
            'injection_step_idx': r.injection_step_idx,
            'correct_step_value': r.correct_step_value,
            'injected_wrong_value': r.injected_wrong_value,
            'injected_text': r.injected_text,
            'model_output': r.model_output,
            'model_answer': r.model_answer,
            'gold_answer': r.gold_answer,
            'outcome': r.outcome,
        })
        if r.hidden_states is not None:
            key = f'{r.problem_id}_{r.injection_point}_{r.error_type}'
            hs_dict[key] = np.array(r.hidden_states)

    csv_path = os.path.join(cfg.output_dir, f'results{suffix}.csv')
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    if hs_dict:
        hs_path = os.path.join(cfg.output_dir, f'hidden_states{suffix}.npy')
        np.save(hs_path, hs_dict, allow_pickle=True)

    print(f'Saved {len(results)} results to {csv_path}')


def run_experiment(
    model,
    tokenizer,
    examples: List[dict],
    cfg: ExperimentConfig,
) -> List[ExperimentResult]:
    os.makedirs(cfg.output_dir, exist_ok=True)
    conditions = [
        (ip, et)
        for ip in cfg.injection_points
        for et in cfg.error_types
    ]
    results: List[ExperimentResult] = []

    pbar = tqdm(total=len(examples) * len(conditions), desc='Running experiment')
    for i, example in enumerate(examples):
        for injection_point, error_type in conditions:
            result = run_single_injection(
                model, tokenizer, example, cfg, injection_point, error_type
            )
            if result is not None:
                results.append(result)
            pbar.update(1)

        if (i + 1) % 20 == 0:
            save_results(results, cfg, checkpoint=True)

    pbar.close()
    return results

## Section 8: Probing Analysis

In [ ]:
def prepare_probe_data(
    results: List[ExperimentResult], cfg: ExperimentConfig
) -> Tuple[Optional[Dict[int, np.ndarray]], Optional[np.ndarray], int]:
    filtered = [
        r for r in results
        if r.hidden_states is not None and r.error_type != 'correct_rephrased'
    ]
    if not filtered:
        print('No examples with hidden states found.')
        return None, None, 0

    labels = np.array([1 if r.outcome == 'full_recovery' else 0 for r in filtered])
    print(f'Probe dataset: {len(filtered)} examples | {labels.mean():.1%} recovery rate')

    n_layers = len(filtered[0].hidden_states)
    layer_features: Dict[int, np.ndarray] = {}
    for li in range(n_layers):
        layer_features[li] = np.array([r.hidden_states[li] for r in filtered])

    return layer_features, labels, n_layers


def train_probes(
    layer_features: Dict[int, np.ndarray],
    labels: np.ndarray,
    n_layers_sample: Optional[int] = None,
) -> Dict[int, float]:
    n_layers = len(layer_features)

    if n_layers_sample is not None:
        layer_indices = list(map(int, np.linspace(0, n_layers - 1, n_layers_sample)))
    elif n_layers > 20:
        layer_indices = list(range(0, n_layers, 4))
    else:
        layer_indices = list(range(n_layers))

    probe_accuracies: Dict[int, float] = {}
    for li in tqdm(layer_indices, desc='Training probes'):
        X = layer_features[li]
        if X.shape[0] < 10:
            continue
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_s, labels, test_size=0.2, random_state=42, stratify=labels
            )
        except ValueError:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_s, labels, test_size=0.2, random_state=42
            )
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X_tr, y_tr)
        probe_accuracies[li] = clf.score(X_te, y_te)

    if probe_accuracies:
        peak = max(probe_accuracies, key=probe_accuracies.get)
        print(f'Peak probe accuracy: {probe_accuracies[peak]:.3f} at layer {peak}')

    return probe_accuracies

## Section 9: Dissociation Analysis

The central novel analysis: identifying cases where the model's hidden states encode the correct answer
but its surface output follows the injected wrong token.

In [ ]:
def analyze_dissociation(
    results: List[ExperimentResult],
    probe_accuracies: Dict[int, float],
    layer_features: Optional[Dict[int, np.ndarray]] = None,
    labels: Optional[np.ndarray] = None,
):
    if not probe_accuracies:
        print('No probe accuracies — skipping dissociation analysis.')
        return

    peak_layer = max(probe_accuracies, key=probe_accuracies.get)
    print(f'Peak probe layer: {peak_layer}, accuracy: {probe_accuracies[peak_layer]:.3f}')

    if layer_features is None or labels is None:
        print('Layer features not provided — cannot compute dissociation.')
        return

    # Train final probe on all data at peak layer
    X = layer_features[peak_layer]
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(X_s, labels)
    proba = clf.predict_proba(X_s)[:, 1]  # P(full_recovery)

    filtered = [
        r for r in results
        if r.hidden_states is not None and r.error_type != 'correct_rephrased'
    ]

    # Dissociation: probe predicts recovery (>0.6) but actual outcome is propagation
    dissociation = [(r, p) for r, p in zip(filtered, proba)
                    if p > 0.6 and r.outcome == 'propagation']

    rate = len(dissociation) / max(len(filtered), 1)
    print(f'\nDissociation cases: {len(dissociation)} / {len(filtered)} ({rate:.1%})')
    print('These are cases where the model internally represented the correct answer '
          'but still followed the wrong surface CoT token.\n')

    for r, p in dissociation[:5]:
        print(f'  Problem {r.problem_id} | {r.injection_point} | {r.error_type}')
        print(f'  Probe confidence (recovery): {p:.3f}')
        print(f'  Gold: {r.gold_answer} | Injected: {r.injected_wrong_value} | Model: {r.model_answer}')
        snippet = (r.injected_text or '')[:200]
        print(f'  Injected text: {snippet}')
        print()

## Section 10: Visualization

In [ ]:
OUTCOME_COLORS = {
    'full_recovery': '#2ecc71',
    'propagation': '#e74c3c',
    'collapse': '#95a5a6',
    'control_correct': '#3498db',
    'control_incorrect': '#e67e22',
}


def _stacked_outcome_bar(ax, df: pd.DataFrame, group_col: str, title: str):
    outcomes = ['full_recovery', 'propagation', 'collapse']
    main_df = df[~df['outcome'].isin(['control_correct', 'control_incorrect'])]
    pivot = (
        main_df.groupby([group_col, 'outcome'])
        .size()
        .unstack(fill_value=0)
    )
    for o in outcomes:
        if o not in pivot.columns:
            pivot[o] = 0
    pivot = pivot[outcomes]
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)
    bottom = np.zeros(len(pivot_pct))
    for o in outcomes:
        ax.bar(
            pivot_pct.index, pivot_pct[o], bottom=bottom,
            color=OUTCOME_COLORS[o], label=o.replace('_', ' ')
        )
        bottom += pivot_pct[o].values
    ax.set_title(title)
    ax.set_ylabel('Proportion')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc='lower right')


def plot_results(
    results: List[ExperimentResult],
    probe_accuracies: Optional[Dict[int, float]],
    cfg: ExperimentConfig,
):
    rows = [{
        'problem_id': r.problem_id, 'difficulty_bin': r.difficulty_bin,
        'injection_point': r.injection_point, 'error_type': r.error_type,
        'outcome': r.outcome, 'model_answer': r.model_answer, 'gold_answer': r.gold_answer,
    } for r in results]
    df = pd.DataFrame(rows)

    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    fig.suptitle(
        f'CoT Injection Results — {cfg.model_name.split("/")[-1]} on {cfg.dataset}',
        fontsize=13, y=0.98
    )

    # Panel 1: outcomes by injection point
    _stacked_outcome_bar(axes[0, 0], df, 'injection_point', 'Outcomes by Injection Point')

    # Panel 2: outcomes by error type
    _stacked_outcome_bar(axes[0, 1], df, 'error_type', 'Outcomes by Error Type')
    axes[0, 1].tick_params(axis='x', rotation=25)

    # Panel 3: recovery vs propagation by difficulty bin
    main_df = df[~df['outcome'].isin(['control_correct', 'control_incorrect'])]
    bin_stats = main_df.groupby('difficulty_bin')['outcome'].value_counts(normalize=True).unstack(fill_value=0)
    for col in ['full_recovery', 'propagation']:
        if col not in bin_stats:
            bin_stats[col] = 0
    x = np.arange(cfg.difficulty_bins)
    w = 0.35
    axes[1, 0].bar(x - w/2, bin_stats.get('full_recovery', [0]*cfg.difficulty_bins),
                   width=w, color=OUTCOME_COLORS['full_recovery'], label='Recovery')
    axes[1, 0].bar(x + w/2, bin_stats.get('propagation', [0]*cfg.difficulty_bins),
                   width=w, color=OUTCOME_COLORS['propagation'], label='Propagation')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(['Easy', 'Medium', 'Hard'][:cfg.difficulty_bins])
    axes[1, 0].set_title('Recovery vs Propagation by Difficulty')
    axes[1, 0].set_ylabel('Rate')
    axes[1, 0].legend()

    # Panel 4: heatmap — recovery rate by injection point × difficulty bin
    pivot_hm = (
        main_df[main_df['outcome'] == 'full_recovery']
        .groupby(['injection_point', 'difficulty_bin'])
        .size()
        .div(
            main_df.groupby(['injection_point', 'difficulty_bin']).size()
        )
        .unstack(fill_value=0)
    )
    sns.heatmap(
        pivot_hm, ax=axes[1, 1], annot=True, fmt='.2f',
        cmap='RdYlGn', vmin=0, vmax=1, linewidths=0.5
    )
    axes[1, 1].set_title('Recovery Rate: Injection Point × Difficulty Bin')

    # Panel 5: control vs injection by injection point
    ctrl = df[df['error_type'] == 'correct_rephrased']
    ctrl_acc = ctrl.groupby('injection_point').apply(
        lambda g: (g['outcome'] == 'control_correct').mean()
    )
    inj_rec = main_df.groupby('injection_point').apply(
        lambda g: (g['outcome'] == 'full_recovery').mean()
    )
    pts = list(cfg.injection_points)
    x = np.arange(len(pts))
    axes[2, 0].bar(x - w/2, [ctrl_acc.get(p, 0) for p in pts], width=w,
                   color='#3498db', label='Control accuracy')
    axes[2, 0].bar(x + w/2, [inj_rec.get(p, 0) for p in pts], width=w,
                   color=OUTCOME_COLORS['full_recovery'], label='Recovery (injected)')
    axes[2, 0].set_xticks(x)
    axes[2, 0].set_xticklabels(pts)
    axes[2, 0].set_title('Control vs Injection Recovery by Point')
    axes[2, 0].set_ylabel('Rate')
    axes[2, 0].legend()

    # Panel 6: probe accuracy by layer
    ax6 = axes[2, 1]
    if probe_accuracies:
        layers = sorted(probe_accuracies.keys())
        accs = [probe_accuracies[l] for l in layers]
        ax6.plot(layers, accs, marker='o', markersize=3, linewidth=1.5, color='#2c3e50')
        ax6.axhline(0.5, linestyle='--', color='gray', label='Chance (0.5)')
        peak = max(probe_accuracies, key=probe_accuracies.get)
        ax6.axhline(probe_accuracies[peak], linestyle=':', color='steelblue',
                    label=f'Peak ({probe_accuracies[peak]:.3f}) @ L{peak}')
        ax6.fill_between(layers, 0.5, accs,
                          where=[a > 0.5 for a in accs], alpha=0.2, color='steelblue')
        ax6.set_xlabel('Layer')
        ax6.set_ylabel('Probe test accuracy')
        ax6.set_title(
            'Probe Accuracy by Layer\n'
            '(>0.5: correct answer recoverable from hidden states even when surface CoT is wrong)'
        )
        ax6.legend(fontsize=8)
    else:
        ax6.text(0.5, 0.5, 'Probe data not available', ha='center', va='center',
                 transform=ax6.transAxes)
        ax6.set_title('Probe Accuracy by Layer')

    plt.tight_layout()
    os.makedirs(cfg.output_dir, exist_ok=True)
    base = os.path.join(cfg.output_dir, 'results_figure')
    fig.savefig(base + '.pdf', bbox_inches='tight')
    fig.savefig(base + '.png', bbox_inches='tight', dpi=150)
    print(f'Figure saved to {base}.pdf / .png')
    plt.show()

## Section 11: Multi-Model Scale Comparison

Load CSV results from multiple model runs (3B and 7B) and compare recovery/propagation rates.

In [ ]:
def compare_across_models(
    result_dir: str,
    dataset: str,
    model_tags: List[str],
):
    """Load per-model CSVs and plot cross-model comparison.
    Each CSV must live at: result_dir/<model_tag>/results.csv
    """
    dfs = []
    for tag in model_tags:
        path = os.path.join(result_dir, tag, 'results.csv')
        if not os.path.exists(path):
            print(f'Missing: {path}')
            continue
        df = pd.read_csv(path)
        df['model'] = tag
        dfs.append(df)

    if not dfs:
        print('No result files found.')
        return

    all_df = pd.concat(dfs, ignore_index=True)
    main_df = all_df[~all_df['outcome'].isin(['control_correct', 'control_incorrect'])]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Multi-Model Comparison — {dataset}')

    # Panel 1: Recovery rate by model × difficulty bin
    rec = (
        main_df.groupby(['model', 'difficulty_bin'])['outcome']
        .apply(lambda g: (g == 'full_recovery').mean())
        .reset_index(name='recovery_rate')
    )
    for tag in model_tags:
        sub = rec[rec['model'] == tag]
        ax1.plot(sub['difficulty_bin'], sub['recovery_rate'], marker='o', label=tag)
    ax1.set_xticks([0, 1, 2])
    ax1.set_xticklabels(['Easy', 'Medium', 'Hard'])
    ax1.set_ylabel('Recovery rate')
    ax1.set_title('Recovery Rate by Model × Difficulty')
    ax1.legend(fontsize=8)

    # Panel 2: Propagation rate by model × injection point
    prop = (
        main_df.groupby(['model', 'injection_point'])['outcome']
        .apply(lambda g: (g == 'propagation').mean())
        .reset_index(name='propagation_rate')
    )
    pts = ['early', 'middle', 'late']
    x = np.arange(len(pts))
    w = 0.8 / max(len(model_tags), 1)
    for i, tag in enumerate(model_tags):
        sub = prop[prop['model'] == tag]
        rates = [sub[sub['injection_point'] == p]['propagation_rate'].values[0]
                 if p in sub['injection_point'].values else 0 for p in pts]
        ax2.bar(x + (i - len(model_tags)/2 + 0.5) * w, rates, width=w, label=tag)
    ax2.set_xticks(x)
    ax2.set_xticklabels(pts)
    ax2.set_ylabel('Propagation rate')
    ax2.set_title('Propagation Rate by Model × Injection Point')
    ax2.legend(fontsize=8)

    plt.tight_layout()
    out = os.path.join(result_dir, f'model_comparison_{dataset}.png')
    fig.savefig(out, bbox_inches='tight', dpi=150)
    print(f'Saved comparison figure to {out}')
    plt.show()


# Usage example (run after completing experiments for both model sizes):
# compare_across_models(
#     result_dir='./results',
#     dataset='gsm8k',
#     model_tags=['Qwen2.5-3B-Instruct', 'Qwen2.5-7B-Instruct'],
# )

## Section 12: Qualitative Inspection

In [ ]:
def show_examples(df: pd.DataFrame, outcome: str, n: int = 3):
    """Print n random examples for a given outcome type."""
    subset = df[df['outcome'] == outcome]
    if subset.empty:
        print(f'No examples with outcome={outcome}')
        return
    sample = subset.sample(min(n, len(subset)), random_state=0)
    print(f'=== Outcome: {outcome} ({len(subset)} total) ===')
    for _, row in sample.iterrows():
        print(f'Problem {int(row["problem_id"])} | '
              f'difficulty_bin={int(row["difficulty_bin"])} | '
              f'{row["injection_point"]} | {row["error_type"]}')
        print(f'  Injected text: {str(row.get("injected_text", ""))[:200]}')
        print(f'  Model output:  {str(row.get("model_output", ""))[:400]}')
        print(f'  Gold: {row["gold_answer"]}  |  Model: {row["model_answer"]}')
        print()

## Main Execution

Run all cells above first, then execute this cell to run the full experiment.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
cfg = ExperimentConfig(
    model_name='Qwen/Qwen2.5-7B-Instruct',  # or 'Qwen/Qwen2.5-3B-Instruct'
    dataset='gsm8k',                          # or 'prontoqa'
    n_problems=200,
    run_probing=True,
    output_dir='./results/7B_gsm8k',
)

# ── Load model ───────────────────────────────────────────────────────────────
model, tokenizer = load_model(cfg)

# ── Load dataset ─────────────────────────────────────────────────────────────
examples = load_examples(cfg)

# ── Run experiment ───────────────────────────────────────────────────────────
results = run_experiment(model, tokenizer, examples, cfg)
save_results(results, cfg)
print(f'Total results: {len(results)}')

# ── Probing analysis ─────────────────────────────────────────────────────────
probe_accuracies = {}
layer_features, labels = None, None
if cfg.run_probing:
    layer_features, labels, n_layers = prepare_probe_data(results, cfg)
    if layer_features is not None:
        probe_accuracies = train_probes(layer_features, labels)

# ── Dissociation analysis ─────────────────────────────────────────────────────
analyze_dissociation(results, probe_accuracies, layer_features, labels)

# ── Visualization ─────────────────────────────────────────────────────────────
plot_results(results, probe_accuracies, cfg)

# ── Qualitative inspection ────────────────────────────────────────────────────
results_df = pd.DataFrame([{
    'problem_id': r.problem_id, 'difficulty_bin': r.difficulty_bin,
    'injection_point': r.injection_point, 'error_type': r.error_type,
    'outcome': r.outcome, 'injected_text': r.injected_text,
    'model_output': r.model_output, 'model_answer': r.model_answer,
    'gold_answer': r.gold_answer,
} for r in results])

for outcome_type in ['full_recovery', 'propagation', 'collapse']:
    show_examples(results_df, outcome_type, n=3)